Task 4:

Download the Invoice OCR Dataset from Kaggle and select at least 5 invoice images to work on.
Use Python OCR library Tesseract to extract text from the invoice images and save the extracted text into .txt files.
From the extracted text, identify important information such as Invoice Number, 
Company Name, Date, Customer Name, and Total Amount, then store the results in a pandas dataframe.

https://www.kaggle.com/datasets/senju14/invoice-ocr

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("senju14/invoice-ocr")

print("Path to dataset files:", path)


c:\Users\AhadS\OneDrive\Desktop\github\Nafath\messy notes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1


In [13]:
import os

root = r"C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1"

for folder, _, files in os.walk(root):
    if files:
        print("\nFolder:", folder)
        print("Sample files:", files[:3])


Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\test\annotations
Sample files: ['X00016469619.json', 'X51005230617.json', 'X51005301659.json']

Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\test\images
Sample files: ['X00016469619.jpg', 'X51005230617.jpg', 'X51005301659.jpg']

Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\train\annotations
Sample files: ['X00016469612.json', 'X00016469620.json', 'X00016469622.json']

Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\train\images
Sample files: ['X00016469612.jpg', 'X00016469620.jpg', 'X00016469622.jpg']

Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\val\annotations
Sample files: ['X00016469623.json', 'X51005255805.json', 'X51005268200.json']

Folder: C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\val\images
Sample files: ['X00016469623.jpg', 'X510052

In [ ]:
import os
import re
import pytesseract
import pandas as pd
from PIL import Image

# =========================
# 1. SET TESSERACT PATH
# =========================
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# =========================
# 2. DATASET PATH
# =========================
image_folder = r"C:\Users\AhadS\.cache\kagglehub\datasets\senju14\invoice-ocr\versions\1\train\images"

# =========================
# 3. STORAGE
# =========================
records = []

# =========================
# 4. LOOP THROUGH IMAGES
# =========================
for file in os.listdir(image_folder):

    if file.lower().endswith((".jpg", ".jpeg", ".png")):

        image_path = os.path.join(image_folder, file)

        # OCR TEXT EXTRACTION
        text = pytesseract.image_to_string(Image.open(image_path))

        # SAVE RAW TEXT FILE
        txt_file = os.path.join(image_folder, file.split(".")[0] + ".txt")
        with open(txt_file, "w", encoding="utf-8") as f:
            f.write(text)

        # =========================
        # 5. EXTRACT INFORMATION
        # =========================

        invoice_no = ""
        date = ""
        total = ""

        inv_match = re.search(r"invoice\s*(?:no|number)?[:#]?\s*([A-Z0-9-]+)", text, re.I)
        if inv_match:
            invoice_no = inv_match.group(1)

        date_match = re.search(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", text)
        if date_match:
            date = date_match.group()

        total_match = re.search(r"(?:total|grand total)[^\d]*([\d,]+\.\d{2})", text, re.I)
        if total_match:
            total = total_match.group(1)

        # =========================
        # 6. BASIC INFO
        # =========================
        lines = [line.strip() for line in text.split("\n") if line.strip()]

        company = lines[0] if len(lines) > 0 else ""
        customer = lines[1] if len(lines) > 1 else ""

        # =========================
        # 7. STORE RECORD
        # =========================
        records.append({
            "File": file,
            "Invoice_Number": invoice_no,
            "Company_Name": company,
            "Date": date,
            "Customer_Name": customer,
            "Total_Amount": total
        })

# =========================
# 8. CREATE DATAFRAME
# =========================
df = pd.DataFrame(records)

# =========================
# 9. DISPLAY OUTPUT
# =========================
from IPython.display import display
display(df)

# =========================
# 10. SAVE RESULT
# =========================
df.to_csv("invoice_results.csv", index=False)

print("DONE ✅ - OCR Completed and CSV saved!")

filling nulls using interpulation 

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(categories=[["A", "B", "C"]])
data['oridinal_col'] = 
oe.fit_transform(data[['oridinal_col']])

In [2]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# sample data
data = pd.DataFrame({
    "ordinal_col": ["A", "B", "C", "A", "B"]
})

oe = OrdinalEncoder(categories=[["A", "B", "C"]])

data['ordinal_col'] = oe.fit_transform(data[['ordinal_col']])

print(data)

   ordinal_col
0          0.0
1          1.0
2          2.0
3          0.0
4          1.0


In [4]:
import pandas as pd
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# 1. Load a real dataset (Titanic)
# This dataset has 'age' (numeric) and 'sex'/'embarked' (categorical)
data = sns.load_dataset('titanic')

# We'll use 'age' as the numeric column and 'who' as the categorical column to match the concept
# Let's subset the data for clarity
df = data[['age', 'who']].copy()
df.columns = ['age', 'category']

print("Original Data (First 5 rows):")
print(df.head())
print("\nMissing values before preprocessing:")
print(df.isnull().sum())

# 2. Define the Numeric Transformer
# Steps: Impute missing values with mean, then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# 3. Define the Categorical Transformer
# Steps: Impute missing values with most frequent, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 4. Combine them into a ColumnTransformer
# Apply numeric_transformer to 'age' and categorical_transformer to 'category'
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, ['age']),
    ('cat', categorical_transformer, ['category'])
])

# 5. Fit and transform the data
preprocessed_data = preprocessor.fit_transform(df)

# 6. Display the result
# Convert back to a DataFrame for easier viewing
# Note: OneHotEncoder creates multiple columns
cat_columns = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(['category'])
all_columns = ['age'] + list(cat_columns)

df_preprocessed = pd.DataFrame(preprocessed_data, columns=all_columns)

print("\nPreprocessed Data (First 5 rows):")
print(df_preprocessed.head())

Original Data (First 5 rows):
    age category
0  22.0      man
1  38.0    woman
2  26.0    woman
3  35.0    woman
4  35.0      man

Missing values before preprocessing:
age         177
category      0
dtype: int64

Preprocessed Data (First 5 rows):
        age  category_child  category_man  category_woman
0 -0.592481             0.0           1.0             0.0
1  0.638789             0.0           0.0             1.0
2 -0.284663             0.0           0.0             1.0
3  0.407926             0.0           0.0             1.0
4  0.407926             0.0           1.0             0.0


In [ ]:
import pandas as pd
import mysql.connector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Step 1: Configure database connection
db_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'password',
    'database': 'your_database_name'
}

# Step 2: Connect to the database
connection = mysql.connector.connect(**db_config)
cursor = connection.cursor()

# Step 3: Fetch all table names
cursor.execute("SHOW TABLES")
tables = [table[0] for table in cursor.fetchall()]

# Step 4: Define preprocessing pipelines

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Step 5: Process each table
for table in tables:
    print(f"Processing table: {table}")

    # Load table into pandas DataFrame
    query = f"SELECT * FROM `{table}`"
    df = pd.read_sql(query, con=connection)

    # Identify numeric and categorical columns
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

    # Skip if no usable data
    if not numeric_columns and not categorical_columns:
        print(f"Table {table} has no numeric or categorical data to process.")
        continue

    # Define ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_columns),
            ('cat', categorical_transformer, categorical_columns)
        ],
        remainder='passthrough'
    )

    # Fit and transform data
    transformed_data = preprocessor.fit_transform(df)

    # Build feature names
    transformed_columns = []

    # numeric columns
    transformed_columns.extend(numeric_columns)

    # categorical encoded columns
    if categorical_columns:
        ohe = preprocessor.named_transformers_['cat'].named_steps['encoder']
        transformed_columns.extend(
            ohe.get_feature_names_out(categorical_columns)
        )

    # passthrough columns
    passthrough_cols = df.columns.difference(numeric_columns + categorical_columns).tolist()
    transformed_columns.extend(passthrough_cols)

    # Convert to DataFrame
    transformed_df = pd.DataFrame(transformed_data, columns=transformed_columns)

    # Create new table
    create_table_query = f"""
    CREATE TABLE `{table}_transformed` (
        {', '.join([f"`{col}` FLOAT" for col in transformed_columns])}
    )
    """

    cursor.execute(f"DROP TABLE IF EXISTS `{table}_transformed`")
    cursor.execute(create_table_query)

    # Insert data
    insert_query = f"""
    INSERT INTO `{table}_transformed`
    VALUES ({', '.join(['%s'] * len(transformed_columns))})
    """

    for row in transformed_df.itertuples(index=False, name=None):
        cursor.execute(insert_query, row)

    print(f"Table {table} transformed and saved as {table}_transformed.")

# Step 6: Commit and close
connection.commit()
cursor.close()
connection.close()

Error: Access denied. Check username/password.


RuntimeError: Database connection not established. Check connection settings and re-run the cell.